# INSTALL DEPENDENCIES

In [ ]:
INITAL_SETUP = True

if INITAL_SETUP:
    %pip install fastkaggle
    %pip install kaggle
    %pip install dotenv
    %pip install ipdb
    %pip install fastai
    %pip install timm
    %pip install -Uqq ddgs fastai
    %pip uninstall -y fastprogress
    %pip install "fastprogress==1.0.3"


In [ ]:
import fastkaggle
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import os
from fastai.tabular import *
from fastai.tabular.all import *
from fastai.vision.all import *
import timm

PATH_LOCAL_PATH_DATA_STORAGE = Path(r'D:\kaggle_data')
COMPETITION_NAME = '104-flowers-garden-of-eden'


# CHECK IF RUNNING ON KAGGLE OR ELSEWHERE

In [ ]:
'Kaggle' if fastkaggle.iskaggle else 'Not Kaggle'

In [ ]:
import torch
import fastai
print(torch.__version__)
print(fastai.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# PREPARE KAGGLE DATA

In [ ]:
FORMAT_SIZE = "jpeg-224x224"

if fastkaggle.iskaggle:
    #path = os.path.join(Path('../input'), COMPETITION_NAME) # generates path containing competition name
    #for dirname, _, filenames in os.walk('/kaggle/input'):
    #    for filename in filenames:
            #print(os.path.join(dirname, filename))
    #        pass
    path = os.path.join('/kaggle/input/datasets/msheriey/104-flowers-garden-of-eden',
                       FORMAT_SIZE) # generates path containing competition name
    
else:
    path = os.path.join(Path(PATH_LOCAL_PATH_DATA_STORAGE), COMPETITION_NAME) # generates path containing competition name
    path = os.path.join(path, 'versions', '1', FORMAT_SIZE) # generates path containing competition name

path
    


In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("msheriey/104-flowers-garden-of-eden")

# print("Path to dataset files:", path)

Check to see if any images are broken and cannot be opened. 

In [ ]:
CHECK_FOR_BROKEN_IMAGINES = False

if CHECK_FOR_BROKEN_IMAGINES:
    failed_train = verify_images(get_image_files(os.path.join(path,'train')))
    print(f"Number of broken images for train: {len(failed_train)}")
    for f in failed_train:
        print(f)

    failed_val = verify_images(get_image_files(os.path.join(path,'val')))
    print(f"Number of broken images for validiation: {len(failed_val)}")
    for f in failed_val:
        print(f)

    failed_test = verify_images(get_image_files(os.path.join(path,'test')))
    print(f"Number of broken images for test: {len(failed_test)}")
    for f in failed_test:
        print(f)

# Check out the images

In [ ]:
dls = ImageDataLoaders.from_folder(
    path,                                # path to images 
    train='train',
    valid='val',
    blocks = (ImageBlock, CategoryBlock),       # specify the type of data
    seed = 42,                           # random seed
    get_y = parent_label,
    item_tfms = Resize(224), # transform the images to 224x224
    batch_tfms=aug_transforms())   # provides a pre-configured, best-practice set of transformations (flips, rotations, zooms, 
                            #brightness changes, warping) with a 75% probability of application.          

dls.show_batch(max_n=20, nrows=5)

In [ ]:
for vocab in dls.vocab:
    print(vocab)

# Vision Learner

In [ ]:
learn = vision_learner(dls, 
                       convnext_tiny, #resnet18,
                       weights=ConvNeXt_Tiny_Weights.DEFAULT,
                       model_dir='/kaggle/working',
                       metrics=[accuracy, error_rate]).to_fp16()

# Fine Tune
We fine tune this model because we want to take advantage of the pretrained model. This will sever the head and retrain the last layer for our specific application. 

In [ ]:
#lrs = learn.lr_find(suggest_funcs=(minimum, steep, valley, slide))

The next line is not necessary for training but does help ensure that the progress bar continues moving. This problem only exists when the python code is executing in Visual Studio Code. 

In [ ]:
if 'TERM_PROGRAM' in os.environ.keys() and os.environ['TERM_PROGRAM'] == 'vscode':
    print("Running in VS Code")
    from IPython.display import clear_output, DisplayHandle
    def update_patch(self, obj):
        clear_output(wait=True)
        self.display(obj)
    DisplayHandle.update = update_patch

In [ ]:
learn.fine_tune(3, base_lr = 0.0006918309954926372 )

# Quick check of the results

In [ ]:
learn.show_results()

# Save model weights and state

In [ ]:
learn.save('trained')

# Cleanup dataset

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)


In [ ]:
interp.plot_top_losses(20, figsize=((13,13)), nrows=8)

In [ ]:
#class_report = interp.print_classification_report()
#class_report.sort_values('f1-score', axis='columns')

In [ ]:
#from fastai.vision.widgets import *
#cleaner = ImageClassifierCleaner(learn)
#cleaner

In [ ]:
## Delete images selected for deletion.
#for index in cleaner.delete():
#    cleaner.fns[index].unlink()

## Relabel images selected for relabeling.
#for index, category in cleaner.change():
#    shutil.move(str(cleaner.fns[index]), path/category)

# Prepare testing data

Load testing data....

In [ ]:
test_files = get_image_files(os.path.join(path,'test'))
print("Found {} test files.".format(len(test_files)))

In [ ]:
dls_test = learn.dls.test_dl(test_files)

# Inference

In [ ]:
preds, _, decoded = learn.get_preds(dl=dls_test, with_decoded=True)

Let's check the results

In [ ]:
preds, decoded
predicted_classes = [learn.dls.vocab[i] for i in preds.argmax(dim=1)]
len(predicted_classes)
print('Length of preds:', len(preds))
print('Length of decoded:', len(decoded))
print('Length of predicted_classes:', len(predicted_classes))


In [ ]:
predicted_classes[:10]


In [ ]:
#learn.show_results(dl=dls_test, max_n=5)
#interp = Interpretation.from_learner(learn, dl=dls_test)
#interp.plot_top_losses(9)
#learn.dls.show_results(dls_test.one_batch(), preds, max_n=9)

In [ ]:
predicted_classes[0]

In [ ]:
if fastkaggle.iskaggle:
    path_submission = '/kaggle/input/competitions/tpu-getting-started/sample_submission.csv'
else:
    path_submission = os.path.join(path, '..', 'sample_submission.csv')

submission_csv = pd.read_csv(path_submission, header=0)
submission_csv.head(10)


# Adding predictions to the submission CSV 

In [ ]:
dls_test.items[:10]

In [ ]:
# Get the original filenames/paths
file_names = [f.name.replace('.jpeg', '') for f in dls_test.items]
file_names[:10]

In [ ]:
# Combine them into a dictionary or list for easy viewing
results = list(zip(file_names, predicted_classes))
results[:10]
d_results = dict(results)

In [ ]:
print(submission_csv.loc[0, 'id'] )
print(submission_csv.loc[0, 'label'] )

In [ ]:
submission_csv['label'] = submission_csv['label'].astype(str)

for index, row in submission_csv.iterrows():
    #print('index', index)
    #print('row id', row['id'])
    #print('row label', row['label'])
    matched_index = d_results.get(row['id'])
    #print('new label', matched_index)
    submission_csv.loc[index, 'label'] = matched_index
    #print('row label after', row['label'])
    # if index>10:
    #     break'

In [ ]:
submission_csv.head(10)

Need to have numbers representing the labels, and not the names of the flowers themselves

In [ ]:
# CLASSES from https://www.kaggle.com/ryanholbrook/petal-helper
CLASSES = ['pink primrose',    'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea',     'wild geranium',     'tiger lily',           'moon orchid',              'bird of paradise', 'monkshood',        'globe thistle',         # 00 - 09
           'snapdragon',       "colt's foot",               'king protea',      'spear thistle', 'yellow iris',       'globe-flower',         'purple coneflower',        'peruvian lily',    'balloon flower',   'giant white arum lily', # 10 - 19
           'fire lily',        'pincushion flower',         'fritillary',       'red ginger',    'grape hyacinth',    'corn poppy',           'prince of wales feathers', 'stemless gentian', 'artichoke',        'sweet william',         # 20 - 29
           'carnation',        'garden phlox',              'love in the mist', 'cosmos',        'alpine sea holly',  'ruby-lipped cattleya', 'cape flower',              'great masterwort', 'siam tulip',       'lenten rose',           # 30 - 39
           'barberton daisy',  'daffodil',                  'sword lily',       'poinsettia',    'bolero deep blue',  'wallflower',           'marigold',                 'buttercup',        'daisy',            'common dandelion',      # 40 - 49
           'petunia',          'wild pansy',                'primula',          'sunflower',     'lilac hibiscus',    'bishop of llandaff',   'gaura',                    'geranium',         'orange dahlia',    'pink-yellow dahlia',    # 50 - 59
           'cautleya spicata', 'japanese anemone',          'black-eyed susan', 'silverbush',    'californian poppy', 'osteospermum',         'spring crocus',            'iris',             'windflower',       'tree poppy',            # 60 - 69
           'gazania',          'azalea',                    'water lily',       'rose',          'thorn apple',       'morning glory',        'passion flower',           'lotus',            'toad lily',        'anthurium',             # 70 - 79
           'frangipani',       'clematis',                  'hibiscus',         'columbine',     'desert-rose',       'tree mallow',          'magnolia',                 'cyclamen ',        'watercress',       'canna lily',            # 80 - 89
           'hippeastrum ',     'bee balm',                  'pink quill',       'foxglove',      'bougainvillea',     'camellia',             'mallow',                   'mexican petunia',  'bromelia',         'blanket flower',        # 90 - 99
           'trumpet creeper',  'blackberry lily',           'common tulip',     'wild rose']                                                                                                                                               # 100 - 103

CLASS_NUMS = list(range(len(CLASSES)))
class_converter = dict(zip(CLASSES, CLASS_NUMS))

In [ ]:
def convert_class_to_num(flower_class):
    flower_number = class_converter.get(flower_class)
    return flower_number 
    
submission_csv['label'] = submission_csv['label'].apply(convert_class_to_num)

In [ ]:
submission_csv.to_csv('submission.csv', index=False)